# P7S amendment A1: evaluation stage (the single run)

**This is the registered test. It is run once.** It reads the evaluation window (2024-07-01 to 2026-10-28) for
the universe and thresholds fixed in the development stage.

1. Run the cell. Nothing needs to be uploaded.
   - If `p7s_dev_outputs.zip` from the development run is in this session, it is used.
   - If not, the notebook first runs the development stage again (development data only, about 20 to 30 minutes)
     and checks that it reproduces the registered development run. If it does not, it stops.
2. The evaluation itself takes another 20 to 30 minutes. If the connection drops during downloads, run the cell
   again: finished downloads are kept, and no statistic is computed until every file is in.
3. Send back the printed output and the two files that download at the end, `p7s_eval_results.zip` and
   `p7s_dev_outputs.zip`.

Use a computer, not a phone: the run is long and a sleeping browser tab disconnects it.
Whatever the verdict, it is reported as it comes out.


In [ ]:
import datetime, hashlib, json, os, sys, time, zipfile

SRC = {"p7s_executor_A1.py": "\"\"\"P7S executor, amendment A1 (P7S_amendment_A1.md). Supersedes p7s_executor.py, which did not implement the\nregistered inference or the registered proximity variable.\n\nPipeline, per symbol, on a complete 5-minute grid:\n  r_t      = ln(P_t / P_{t-1})                          5-minute log return\n  Pi_t     = V_t / max_{t-30d < s <= t} V_s             V = open-interest value; needs 30 full days of history\n  response = sign(r_t) * ln(P_{t+12} / P_t) / sigma60   60-minute continuation, in units of the symbol's\n                                                        development-period sd of 60-minute returns\n  calm     : q50 <= |r_t|/sigma <= q80, and no stress   thresholds from the development period, per symbol\n             bar in the previous 60 minutes\n  stress   : |r_t|/sigma >= q99\n  high/low : Pi_t >= pi_hi / Pi_t <= pi_lo              development terciles of Pi, per symbol\nFor each symbol: beta_calm = mean(response | calm, high) - mean(response | calm, low); beta_stress likewise.\nStatistics: mean over symbols of beta_calm, of beta_stress, and DiD = their difference.\nInference:\n  intervals : stationary block bootstrap over dates (mean block 5 days), the same resampled dates for every symbol;\n  H1        : two one-sided tests at 5% (the 90% bootstrap interval of mean beta_calm inside (-E, E));\n  H2        : a time-shift randomization test. The high/low state series of every symbol is shifted circularly by\n              the same k days against the returns, for k = step, 2*step, ..., n_days - step, and the DiD is\n              recomputed. n_days must be a multiple of step, so the shifts and the unshifted series form a cyclic\n              group and the test is exact when the state series is independent of the return series and both are\n              stationary. p = (1 + #{|DiD_k| >= |DiD|}) / (n_days / step). The smallest possible p is step / n_days.\n              The percentile bootstrap for the DiD rejected a true linear null in 8 to 9% of simulated data sets\n              (validate_p7s_A1); the shift test replaces it for H2. The bootstrap p-value is kept as a descriptive\n              output.\nDecision (spec section 4): verdict supportive / inconsistent / inconclusive.\n\nNothing in this file downloads data. Thresholds are fitted on development data only and passed in.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nSEED = 20261108\nB_DEFAULT = 9999\nMEAN_BLOCK_DAYS = 5.0\nE_MARGIN = 0.10            # sd units of the 60-minute return; fixed a priori (amendment A1, section 3)\nH_BARS = 12                # 60 minutes\nBARS_PER_DAY = 288\nTRAIL_BARS = 30 * BARS_PER_DAY\nCALM_GAP_BARS = 12         # a calm bar has no stress bar in the previous 60 minutes\nSHIFT_STEP_DAYS = 5        # spacing of the circular shifts in the randomization test; must divide n_days\nMIN_EVENTS = 20            # a symbol enters only with >= 20 events in each of the four cells\nCELLS = (\"calm_high\", \"calm_low\", \"stress_high\", \"stress_low\")\n\n\ndef log_returns(close: np.ndarray) -> np.ndarray:\n    r = np.full(len(close), np.nan)\n    with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n        r[1:] = np.log(close[1:] / close[:-1])\n    return r\n\n\ndef forward_return(close: np.ndarray, h: int = H_BARS) -> np.ndarray:\n    f = np.full(len(close), np.nan)\n    with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n        f[:-h] = np.log(close[h:] / close[:-h])\n    return f\n\n\ndef proximity(oi_value: np.ndarray, trail: int = TRAIL_BARS) -> np.ndarray:\n    \"\"\"Pi_t = V_t / max over the trailing `trail` bars (including t). NaN until `trail` bars of history exist, and\n    NaN if fewer than 95% of the bars in the window are observed.\"\"\"\n    s = pd.Series(oi_value, dtype=\"float64\")\n    mx = s.rolling(trail, min_periods=int(0.95 * trail)).max().to_numpy()\n    pi = oi_value / mx\n    pi[: trail - 1] = np.nan\n    return pi\n\n\ndef trailing_rv(r: np.ndarray, window: int = BARS_PER_DAY) -> np.ndarray:\n    \"\"\"Realized volatility of 5-minute returns over the previous `window` bars (excluding t).\"\"\"\n    s = pd.Series(r ** 2, dtype=\"float64\").rolling(window, min_periods=int(0.9 * window)).mean().shift(1)\n    return np.sqrt(s.to_numpy())\n\n\ndef fit_thresholds(close: np.ndarray, state_var: np.ndarray, dev_mask: np.ndarray, rv_normalize: bool = False) -> dict:\n    \"\"\"Thresholds for one symbol from development bars only. `state_var` is Pi (or the variant's state variable).\"\"\"\n    r = log_returns(close); f = forward_return(close)\n    sigma = float(np.nanstd(r[dev_mask]))\n    sigma60 = float(np.nanstd(f[dev_mask]))\n    z = np.abs(r) / (trailing_rv(r) if rv_normalize else sigma)\n    zd = z[dev_mask & np.isfinite(z)]\n    sd = state_var[dev_mask & np.isfinite(state_var)]\n    q50, q80, q99 = (float(x) for x in np.percentile(zd, [50, 80, 99]))\n    lo, hi = (float(x) for x in np.percentile(sd, [100 / 3, 200 / 3]))\n    return {\"sigma\": sigma, \"sigma60\": sigma60, \"q50\": q50, \"q80\": q80, \"q99\": q99, \"state_lo\": lo, \"state_hi\": hi,\n            \"n_dev_bars\": int(len(zd)), \"rv_normalize\": bool(rv_normalize)}\n\n\ndef symbol_cells(close: np.ndarray, state_var: np.ndarray, day_id: np.ndarray, eval_mask: np.ndarray, thr: dict,\n                 n_days: int) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Per-day sums and counts of the response in the four cells, for one symbol. Returns (S, N), each (n_days, 4).\n    `day_id` maps each bar to 0..n_days-1 on evaluation bars; an event belongs to the day of its initiating bar.\"\"\"\n    r = log_returns(close); f = forward_return(close)\n    z = np.abs(r) / (trailing_rv(r) if thr[\"rv_normalize\"] else thr[\"sigma\"])\n    resp = np.sign(r) * f / thr[\"sigma60\"]\n    ok = eval_mask & np.isfinite(z) & np.isfinite(resp) & np.isfinite(state_var) & (r != 0)\n    is_stress = np.where(np.isfinite(z), z >= thr[\"q99\"], False)\n    recent = pd.Series(is_stress.astype(float)).rolling(CALM_GAP_BARS, min_periods=1).max().shift(1).fillna(0.0).to_numpy() > 0\n    calm = ok & (z >= thr[\"q50\"]) & (z <= thr[\"q80\"]) & ~recent\n    stress = ok & is_stress\n    high = state_var >= thr[\"state_hi\"]\n    low = state_var <= thr[\"state_lo\"]\n    S = np.zeros((n_days, 4)); N = np.zeros((n_days, 4))\n    for k, m in enumerate((calm & high, calm & low, stress & high, stress & low)):\n        np.add.at(S[:, k], day_id[m], resp[m]); np.add.at(N[:, k], day_id[m], 1.0)\n    return S, N\n\n\ndef symbol_bars(close: np.ndarray, state_var: np.ndarray, eval_mask: np.ndarray, thr: dict) -> dict:\n    \"\"\"Bar-level arrays on evaluation bars, in time order, for the time-shift test: response, move type\n    (1 calm, 2 stress, 0 neither) and state (+1 high, -1 low, 0 neither or missing).\"\"\"\n    r = log_returns(close); f = forward_return(close)\n    z = np.abs(r) / (trailing_rv(r) if thr[\"rv_normalize\"] else thr[\"sigma\"])\n    resp = np.sign(r) * f / thr[\"sigma60\"]\n    ok = np.isfinite(z) & np.isfinite(resp) & (r != 0)\n    is_stress = np.where(np.isfinite(z), z >= thr[\"q99\"], False)\n    recent = pd.Series(is_stress.astype(float)).rolling(CALM_GAP_BARS, min_periods=1).max().shift(1).fillna(0.0).to_numpy() > 0\n    move = np.zeros(len(close), dtype=np.int8)\n    move[ok & (z >= thr[\"q50\"]) & (z <= thr[\"q80\"]) & ~recent] = 1\n    move[ok & is_stress] = 2\n    state = np.zeros(len(close), dtype=np.int8)\n    state[np.isfinite(state_var) & (state_var >= thr[\"state_hi\"])] = 1\n    state[np.isfinite(state_var) & (state_var <= thr[\"state_lo\"])] = -1\n    return {\"resp\": np.where(ok, resp, 0.0)[eval_mask], \"move\": move[eval_mask], \"state\": state[eval_mask]}\n\n\ndef _did_bars(bars: list[dict], shift_bars: int) -> tuple[float, float]:\n    \"\"\"Mean over symbols of beta_calm and the DiD, with every state series rolled forward by `shift_bars` bars.\"\"\"\n    bc, bs = [], []\n    for b in bars:\n        st = np.roll(b[\"state\"], shift_bars) if shift_bars else b[\"state\"]\n        m = []\n        for mv in (1, 2):\n            for sg in (1, -1):\n                sel = (b[\"move\"] == mv) & (st == sg)\n                m.append(b[\"resp\"][sel].mean() if sel.any() else np.nan)\n        bc.append(m[0] - m[1]); bs.append(m[2] - m[3])\n    bc, bs = np.array(bc), np.array(bs)\n    return float(np.nanmean(bc)), float(np.nanmean(bs) - np.nanmean(bc))\n\n\ndef shift_test(bars: list[dict], n_days: int, step_days: int = SHIFT_STEP_DAYS) -> dict:\n    \"\"\"Time-shift randomization p-value for the DiD. `bars` holds only the symbols used by `evaluate`.\n    The shifts step, 2*step, ..., n_days - step and the unshifted series form a cyclic group.\"\"\"\n    if n_days % step_days != 0:\n        raise ValueError(f\"n_days={n_days} must be a multiple of step_days={step_days}\")\n    _, did = _did_bars(bars, 0)\n    shifts = list(range(step_days, n_days, step_days))\n    null_all = np.array([_did_bars(bars, k * BARS_PER_DAY)[1] for k in shifts])\n    null = null_all[np.isfinite(null_all)]\n    p = float((1 + np.sum(np.abs(null) >= abs(did))) / (1 + len(null)))\n    return {\"did\": did, \"p_shift\": p, \"n_shifts\": int(len(null)), \"step_days\": int(step_days), \"null_sd\": float(np.std(null)),\n            \"null_quantiles_2.5_50_97.5\": [float(x) for x in np.percentile(null, [2.5, 50, 97.5])],\n            \"shift_days\": [int(k) for k in shifts],\n            \"did_by_shift\": [float(x) if np.isfinite(x) else None for x in null_all]}\n\n\ndef stationary_weights(n_days: int, B: int, mean_block: float, rng: np.random.Generator) -> np.ndarray:\n    \"\"\"B x n_days matrix of day multiplicities from the stationary block bootstrap (circular).\"\"\"\n    p = 1.0 / mean_block\n    W = np.zeros((B, n_days))\n    for b in range(B):\n        new = rng.random(n_days) < p; new[0] = True\n        starts = rng.integers(n_days, size=n_days)\n        idx = np.empty(n_days, dtype=np.int64); cur = 0\n        for i in range(n_days):\n            cur = starts[i] if new[i] else (cur + 1) % n_days\n            idx[i] = cur\n        W[b] = np.bincount(idx, minlength=n_days)\n    return W\n\n\ndef _betas(S: np.ndarray, N: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"S, N: (..., n_symbols, 4) aggregated over days. Returns beta_calm, beta_stress per symbol (NaN if a cell is empty).\"\"\"\n    with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n        m = np.where(N > 0, S / N, np.nan)\n    return m[..., 0] - m[..., 1], m[..., 2] - m[..., 3]\n\n\ndef evaluate(S: np.ndarray, N: np.ndarray, symbols: list[str], bars: list[dict] | None = None, B: int = B_DEFAULT,\n             seed: int = SEED, mean_block: float = MEAN_BLOCK_DAYS, e_margin: float = E_MARGIN,\n             min_events: int = MIN_EVENTS, shift_step_days: int = SHIFT_STEP_DAYS) -> dict:\n    \"\"\"S, N: (n_symbols, n_days, 4); bars: per-symbol output of `symbol_bars` (needed for H2 and the verdict).\"\"\"\n    n_sym, n_days, _ = S.shape\n    tot = N.sum(1)\n    keep = (tot >= min_events).all(1)\n    Sk, Nk = S[keep], N[keep]\n    bc, bs = _betas(Sk.sum(1), Nk.sum(1))\n    mean_calm, mean_stress = float(np.mean(bc)), float(np.mean(bs)); did = mean_stress - mean_calm\n    rng = np.random.default_rng(seed)\n    boot_calm = np.empty(B); boot_did = np.empty(B)\n    chunk = 500\n    for a in range(0, B, chunk):\n        W = stationary_weights(n_days, min(chunk, B - a), mean_block, rng)\n        Sb = np.einsum(\"bd,sdc->bsc\", W, Sk); Nb = np.einsum(\"bd,sdc->bsc\", W, Nk)\n        c, s = _betas(Sb, Nb)\n        boot_calm[a:a + len(W)] = np.nanmean(c, axis=1)\n        boot_did[a:a + len(W)] = np.nanmean(s, axis=1) - np.nanmean(c, axis=1)\n    finite = np.isfinite(boot_calm) & np.isfinite(boot_did)     # a draw with an empty cell for every symbol is dropped\n    n_dropped = int((~finite).sum())\n    boot_calm, boot_did = boot_calm[finite], boot_did[finite]\n    ci90_calm = [float(x) for x in np.percentile(boot_calm, [5, 95])]\n    ci95_calm = [float(x) for x in np.percentile(boot_calm, [2.5, 97.5])]\n    ci95_did = [float(x) for x in np.percentile(boot_did, [2.5, 97.5])]\n    h1 = (ci90_calm[0] > -e_margin) and (ci90_calm[1] < e_margin)\n    p_boot = float(min(1.0, 2.0 * min(np.mean(boot_did <= 0), np.mean(boot_did >= 0))))\n    st = shift_test([b for b, k in zip(bars, keep) if k], n_days, step_days=shift_step_days) if bars is not None else None\n    p_h2 = st[\"p_shift\"] if st is not None else float(\"nan\")\n    h2 = bool(st is not None and p_h2 < 0.05)\n    if h1 and h2:\n        verdict = \"supportive\"\n    elif abs(mean_calm) >= e_margin:\n        verdict = \"inconsistent\"\n    else:\n        verdict = \"inconclusive\"\n    return {\"spec\": \"P7S\", \"amendment\": \"A1\", \"n_symbols_total\": int(n_sym), \"n_symbols_used\": int(keep.sum()),\n            \"symbols_excluded\": [s for s, k in zip(symbols, keep) if not k], \"n_days\": int(n_days),\n            \"events_per_cell_total\": {c: int(v) for c, v in zip(CELLS, Nk.sum((0, 1)))},\n            \"beta_calm_mean\": mean_calm, \"beta_calm_ci90\": ci90_calm, \"beta_calm_ci95\": ci95_calm,\n            \"beta_stress_mean\": mean_stress, \"did\": did, \"did_ci95\": ci95_did,\n            \"se_beta_calm\": float(np.std(boot_calm)), \"se_did\": float(np.std(boot_did)),\n            \"share_symbols_beta_stress_positive\": float(np.mean(bs > 0)),\n            \"h1_calm_silence\": bool(h1), \"p_h2_stress_reveals\": p_h2, \"h2_stress_reveals\": bool(h2),\n            \"shift_test\": st, \"p_did_percentile_bootstrap_descriptive\": p_boot,\n            \"verdict\": verdict, \"E_margin\": e_margin, \"B\": int(B), \"bootstrap_draws_dropped\": n_dropped,\n            \"mean_block_days\": mean_block, \"seed\": int(seed),\n            \"min_events\": int(min_events),\n            \"per_symbol\": {s: {\"beta_calm\": float(a), \"beta_stress\": float(b)} for s, a, b in\n                           zip([s for s, k in zip(symbols, keep) if k], bc, bs)}}\n\n\ndef blinded_precision(S: np.ndarray, N: np.ndarray, symbols: list[str], **kw) -> dict:\n    \"\"\"Design-stage output on development data: standard errors and event counts only. No point estimate, interval\n    or verdict is returned.\"\"\"\n    r = evaluate(S, N, symbols, bars=None, **kw)\n    return {k: r[k] for k in (\"n_symbols_used\", \"symbols_excluded\", \"n_days\", \"events_per_cell_total\", \"se_beta_calm\",\n                              \"se_did\", \"E_margin\", \"B\", \"mean_block_days\")}\n", "p7s_data.py": "\"\"\"P7S data layer (amendment A1): archive addresses, a cached downloader, parsers for the daily zip files and\nassembly of one symbol on a complete 5-minute grid. This file computes no statistic.\n\nFile layout (schema probe of the burned day 2024-01-02, results/p7s_schema_probe_results.zip):\n  klines 5m : open_time [ms], open, high, low, close, volume, close_time, quote_volume, count, ...   288 rows a day\n  metrics   : create_time ['YYYY-MM-DD HH:MM:SS', UTC], symbol, sum_open_interest, sum_open_interest_value,\n              count_toptrader_long_short_ratio, sum_toptrader_long_short_ratio, ...                   288 rows a day\nTiming convention: the bar with open time t carries the return over [t, t + 5 min). Its state variables are the\nmetrics snapshot stamped t, which is taken before that return. A missing snapshot is filled forward for at most\n30 minutes.\n\"\"\"\nfrom __future__ import annotations\n\nimport datetime as dt\nimport io\nimport os\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport xml.etree.ElementTree as ET\nimport zipfile\nfrom concurrent.futures import ThreadPoolExecutor\n\nimport numpy as np\nimport pandas as pd\n\nROOT = \"https://data.binance.vision/data/futures/um/daily\"\nS3 = \"https://s3-ap-northeast-1.amazonaws.com/data.binance.vision\"\nHEADERS = {\"User-Agent\": \"Mozilla/5.0 (research data download)\"}\nBAR_MS = 300_000\nBARS_PER_DAY = 288\nFFILL_BARS = 6\nKL_COLS = [\"open_time\", \"open\", \"high\", \"low\", \"close\", \"volume\", \"close_time\", \"quote_volume\", \"count\",\n           \"taker_buy_volume\", \"taker_buy_quote_volume\", \"ignore\"]\n# Excluded by name before ranking: bases that are stablecoins or fiat, and index products.\nEXCLUDED_BASES = (\"USDC\", \"BUSD\", \"TUSD\", \"FDUSD\", \"USDP\", \"DAI\", \"EUR\", \"BTCDOM\", \"DEFI\", \"FOOTBALL\", \"BLUEBIRD\")\n\n\ndef days(start: str, end: str) -> list[str]:\n    a, b = dt.date.fromisoformat(start), dt.date.fromisoformat(end)\n    return [(a + dt.timedelta(days=i)).isoformat() for i in range((b - a).days + 1)]\n\n\ndef shift_day(day: str, n: int) -> str:\n    return (dt.date.fromisoformat(day) + dt.timedelta(days=n)).isoformat()\n\n\ndef day_start_ms(day: str) -> int:\n    return int(dt.datetime.fromisoformat(day).replace(tzinfo=dt.timezone.utc).timestamp() * 1000)\n\n\ndef kline_url(sym: str, day: str) -> str:\n    q = urllib.parse.quote(sym)\n    return f\"{ROOT}/klines/{q}/5m/{q}-5m-{day}.zip\"\n\n\ndef metrics_url(sym: str, day: str) -> str:\n    q = urllib.parse.quote(sym)\n    return f\"{ROOT}/metrics/{q}/{q}-metrics-{day}.zip\"\n\n\ndef candidate_symbols(folders: list[str]) -> list[str]:\n    \"\"\"USDT-margined perpetuals from the archive's folder names, after the exclusions by name.\"\"\"\n    out = []\n    for s in folders:\n        if not s.isascii() or not s.endswith(\"USDT\") or \"_\" in s:\n            continue\n        if s[:-4] in EXCLUDED_BASES:\n            continue\n        out.append(s)\n    return sorted(set(out))\n\n\ndef _read_zip_csv(raw: bytes, default_cols: list[str] | None) -> pd.DataFrame:\n    with zipfile.ZipFile(io.BytesIO(raw)) as z:\n        names = [n for n in z.namelist() if n.endswith(\".csv\")]\n        if len(names) != 1:\n            raise ValueError(f\"expected one csv in the zip, found {names}\")\n        txt = z.read(names[0]).decode(\"utf-8\")\n    first = txt.split(\"\\n\", 1)[0].split(\",\")[0].strip()\n    has_header = any(c.isalpha() for c in first)\n    df = pd.read_csv(io.StringIO(txt), header=0 if has_header else None, float_precision=\"round_trip\")\n    if not has_header:\n        if default_cols is None or df.shape[1] > len(default_cols):\n            raise ValueError(\"file has no header and no default column names apply\")\n        df.columns = default_cols[: df.shape[1]]\n    df.columns = [str(c).strip().lower() for c in df.columns]\n    return df\n\n\ndef parse_klines(raw: bytes) -> pd.DataFrame:\n    \"\"\"Columns: open_ms (int64, milliseconds), close, quote_volume. Microsecond stamps are converted.\"\"\"\n    df = _read_zip_csv(raw, KL_COLS)\n    t = pd.to_numeric(df[\"open_time\"], errors=\"coerce\").astype(\"int64\").to_numpy()\n    if len(t) and t.max() > 10 ** 14:                           # microseconds\n        t = t // 1000\n    out = pd.DataFrame({\"open_ms\": t,\n                        \"close\": pd.to_numeric(df[\"close\"], errors=\"coerce\").to_numpy(dtype=\"float64\"),\n                        \"quote_volume\": pd.to_numeric(df[\"quote_volume\"], errors=\"coerce\").to_numpy(dtype=\"float64\")})\n    return out\n\n\ndef parse_metrics(raw: bytes) -> pd.DataFrame:\n    \"\"\"Columns: ts_ms (stamp floored to the 5-minute grid), oi_value, top_ratio.\"\"\"\n    df = _read_zip_csv(raw, None)\n    ts = pd.to_datetime(df[\"create_time\"], utc=True, errors=\"coerce\")\n    ok = ts.notna().to_numpy()\n    ms = ((ts[ok] - pd.Timestamp(\"1970-01-01\", tz=\"UTC\")) // pd.Timedelta(milliseconds=1)).to_numpy(dtype=\"int64\")\n    return pd.DataFrame({\n        \"ts_ms\": (ms // BAR_MS) * BAR_MS,\n        \"oi_value\": pd.to_numeric(df[\"sum_open_interest_value\"], errors=\"coerce\").to_numpy(dtype=\"float64\")[ok],\n        \"top_ratio\": pd.to_numeric(df[\"sum_toptrader_long_short_ratio\"], errors=\"coerce\").to_numpy(dtype=\"float64\")[ok]})\n\n\ndef assemble(day_list: list[str], klines: dict[str, bytes | None], metrics: dict[str, bytes | None]) -> dict:\n    \"\"\"One symbol on the complete grid of `day_list` (consecutive days). Missing files or bars are NaN.\n    Returns close, quote_volume, oi_value, top_ratio (each len(day_list) * 288) and counts of what was missing.\"\"\"\n    n = len(day_list) * BARS_PER_DAY\n    t0 = day_start_ms(day_list[0])\n    assert day_start_ms(day_list[-1]) - t0 == (len(day_list) - 1) * BARS_PER_DAY * BAR_MS, \"days must be consecutive\"\n    close = np.full(n, np.nan); qv = np.full(n, np.nan); oi = np.full(n, np.nan); top = np.full(n, np.nan)\n    miss_k = miss_m = 0\n    for d in day_list:\n        raw = klines.get(d)\n        if raw is None:\n            miss_k += 1\n        else:\n            k = parse_klines(raw)\n            idx = (k[\"open_ms\"].to_numpy() - t0) // BAR_MS\n            ok = (idx >= 0) & (idx < n) & ((k[\"open_ms\"].to_numpy() - t0) % BAR_MS == 0)\n            close[idx[ok]] = k[\"close\"].to_numpy()[ok]; qv[idx[ok]] = k[\"quote_volume\"].to_numpy()[ok]\n        raw = metrics.get(d)\n        if raw is None:\n            miss_m += 1\n        else:\n            m = parse_metrics(raw)\n            idx = (m[\"ts_ms\"].to_numpy() - t0) // BAR_MS\n            ok = (idx >= 0) & (idx < n)\n            oi[idx[ok]] = m[\"oi_value\"].to_numpy()[ok]; top[idx[ok]] = m[\"top_ratio\"].to_numpy()[ok]\n    close[~(close > 0)] = np.nan\n    oi[~(oi > 0)] = np.nan\n    top[~(top > 0)] = np.nan\n    raw_oi_bars = int(np.isfinite(oi).sum())\n    oi = pd.Series(oi).ffill(limit=FFILL_BARS).to_numpy()\n    top = pd.Series(top).ffill(limit=FFILL_BARS).to_numpy()\n    return {\"close\": close, \"quote_volume\": qv, \"oi_value\": oi, \"top_ratio\": top,\n            \"missing_kline_files\": miss_k, \"missing_metrics_files\": miss_m,\n            \"kline_bar_coverage\": float(np.isfinite(close).mean()), \"metrics_bar_coverage\": raw_oi_bars / n}\n\n\ndef parse_listing(xml_text: str) -> tuple[list[str], str | None]:\n    \"\"\"(folder names, next marker or None) from one page of the bucket listing.\"\"\"\n    root = ET.fromstring(xml_text)\n    ns = root.tag.split(\"}\")[0] + \"}\" if root.tag.startswith(\"{\") else \"\"\n    names = [e.find(ns + \"Prefix\").text for e in root.findall(ns + \"CommonPrefixes\")]\n    trunc = root.find(ns + \"IsTruncated\")\n    nxt = root.find(ns + \"NextMarker\")\n    more = trunc is not None and trunc.text == \"true\"\n    return names, ((nxt.text if nxt is not None else (names[-1] if names else None)) if more else None)\n\n\ndef listing_has_key(xml_text: str, key: str) -> bool:\n    \"\"\"True if a bucket listing page names exactly this file.\"\"\"\n    root = ET.fromstring(xml_text)\n    ns = root.tag.split(\"}\")[0] + \"}\" if root.tag.startswith(\"{\") else \"\"\n    return any(e.find(ns + \"Key\") is not None and e.find(ns + \"Key\").text == key for e in root.findall(ns + \"Contents\"))\n\n\nclass Archive:\n    \"\"\"Downloader with a disk cache. `get` returns the file's bytes, or None if the archive does not have the file:\n    a 404 answer, or a 403 answer for a file that the bucket listing does not name. Any other failure is retried\n    and then raised, so a network problem is never read as a missing file.\"\"\"\n\n    def __init__(self, cache_dir: str, threads: int = 32, retries: int = 6):\n        self.cache_dir, self.threads, self.retries = cache_dir, threads, retries\n        self.n_downloaded = self.n_cached = self.n_absent = 0\n\n    def _path(self, url: str) -> str:\n        return os.path.join(self.cache_dir, urllib.parse.unquote(url.split(\"/data/futures/um/daily/\")[1]))\n\n    def get(self, url: str) -> bytes | None:\n        p = self._path(url)\n        if os.path.exists(p):\n            self.n_cached += 1\n            return open(p, \"rb\").read()\n        if os.path.exists(p + \".404\"):\n            self.n_absent += 1\n            return None\n        os.makedirs(os.path.dirname(p), exist_ok=True)\n        last = None\n        for attempt in range(self.retries):\n            try:\n                with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=60) as r:\n                    raw = r.read()\n                with zipfile.ZipFile(io.BytesIO(raw)) as z:      # a truncated download fails here and is retried\n                    z.namelist()\n                tmp = p + \".part\"\n                open(tmp, \"wb\").write(raw); os.replace(tmp, p)\n                self.n_downloaded += 1\n                return raw\n            except urllib.error.HTTPError as e:\n                if e.code == 404 or (e.code == 403 and not self._listed(url)):\n                    open(p + \".404\", \"w\").close()\n                    self.n_absent += 1\n                    return None\n                last = e\n            except Exception as e:                               # noqa: BLE001\n                last = e\n            time.sleep(min(30, 2 ** attempt))\n        raise RuntimeError(f\"download failed after {self.retries} tries: {url}: {last!r}\")\n\n    def _listed(self, url: str) -> bool:\n        \"\"\"Asks the bucket listing whether the file exists. A failure of the listing request raises.\"\"\"\n        key = urllib.parse.unquote(url.split(\"data.binance.vision/\")[1])\n        q = f\"{S3}?prefix={urllib.parse.quote(key)}\"\n        with urllib.request.urlopen(urllib.request.Request(q, headers=HEADERS), timeout=60) as r:\n            return listing_has_key(r.read().decode(), key)\n\n    def get_many(self, urls: list[str]) -> dict[str, bytes | None]:\n        with ThreadPoolExecutor(self.threads) as pool:\n            return dict(zip(urls, pool.map(self.get, urls)))\n\n    def list_symbol_folders(self) -> list[str]:\n        prefix = \"data/futures/um/daily/klines/\"\n        out, marker = [], None\n        for _ in range(100):\n            url = f\"{S3}?delimiter=/&prefix={urllib.parse.quote(prefix)}\"\n            if marker:\n                url += \"&marker=\" + urllib.parse.quote(marker)\n            with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=60) as r:\n                names, marker = parse_listing(r.read().decode())\n            out += names\n            if marker is None:\n                break\n        return [n[len(prefix):].strip(\"/\") for n in out]\n", "p7s_pipeline_A1.py": "\"\"\"P7S pipeline, amendment A1: the development stage and the evaluation stage.\n\nrun_development : draws the universe by rule from development data, fits every threshold on development data,\n                  and returns a blinded precision check (standard errors and event counts only).\nrun_evaluation  : the single evaluation run. It downloads first, applies a gate that uses event counts only,\n                  and only then computes the statistics with p7s_executor_A1.\nBoth take an `arch` object with get_many(urls) -> {url: bytes or None}, so they can be tested without a network.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\nimport p7s_data as pdz\nimport p7s_executor_A1 as ex\n\nBPD = ex.BARS_PER_DAY\nHISTORY_DAYS = 30            # days of history before the first development or evaluation day (for Pi)\nMIN_SYMBOLS_GATE = 20        # the evaluation stops before any statistic if fewer symbols have 20 events per cell\nSENS_B = 999                 # bootstrap draws for the descriptive sensitivity checks\n\nCFG_PRIMARY = {\"name\": \"primary_retrospective\", \"warmup_start\": \"2023-12-04\", \"dev_start\": \"2024-01-03\",\n               \"dev_end\": \"2024-06-30\", \"eval_start\": \"2024-07-01\", \"eval_end\": \"2026-10-28\",\n               \"shift_step_days\": 5, \"n_universe\": 30}\nCFG_REPLICATION = {\"name\": \"prospective_replication\", \"warmup_start\": \"2026-07-02\", \"dev_start\": \"2026-08-01\",\n                   \"dev_end\": \"2026-10-31\", \"eval_start\": \"2026-11-16\", \"eval_end\": \"2027-02-14\",\n                   \"shift_step_days\": 1, \"n_universe\": 30}\n\n\ndef _fetch(arch, sym: str, day_list: list[str], metrics: bool = True):\n    ku = [pdz.kline_url(sym, d) for d in day_list]\n    mu = [pdz.metrics_url(sym, d) for d in day_list] if metrics else []\n    got = arch.get_many(ku + mu)\n    return ({d: got[u] for d, u in zip(day_list, ku)}, {d: got[u] for d, u in zip(day_list, mu)})\n\n\ndef _states(arr: dict) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"(Pi, top-trader state). The top-trader state is |ln(top-trader long/short ratio)|.\"\"\"\n    pi = ex.proximity(arr[\"oi_value\"])\n    with np.errstate(invalid=\"ignore\", divide=\"ignore\"):\n        top = np.abs(np.log(arr[\"top_ratio\"]))\n    return pi, top\n\n\ndef _volume_tercile(qv: np.ndarray, cuts: list[float]) -> np.ndarray:\n    \"\"\"0, 1, 2 for the development terciles of 5-minute quote volume; -1 where volume is missing.\"\"\"\n    t = np.full(len(qv), -1, dtype=np.int8)\n    ok = np.isfinite(qv)\n    t[ok & (qv <= cuts[0])] = 0\n    t[ok & (qv > cuts[0]) & (qv <= cuts[1])] = 1\n    t[ok & (qv > cuts[1])] = 2\n    return t\n\n\ndef run_development(cfg: dict, arch, folders: list[str], log=print, B: int = SENS_B) -> dict:\n    warm = pdz.days(cfg[\"warmup_start\"], pdz.shift_day(cfg[\"dev_start\"], -1))\n    dev = pdz.days(cfg[\"dev_start\"], cfg[\"dev_end\"])\n    assert len(warm) == HISTORY_DAYS, f\"warm-up must be {HISTORY_DAYS} days, got {len(warm)}\"\n    all_days = warm + dev\n    cands = pdz.candidate_symbols(folders)\n    log(f\"{len(folders)} folders listed; {len(cands)} candidate USDT perpetuals after the exclusions by name\")\n\n    # 1. screen: both files on the first warm-up day and on the last development day\n    probe = [warm[0], dev[-1]]\n    urls = [f(s, d) for s in cands for d in probe for f in (pdz.kline_url, pdz.metrics_url)]\n    got = arch.get_many(urls)\n    screened = [s for s in cands if all(got[f(s, d)] is not None for d in probe for f in (pdz.kline_url, pdz.metrics_url))]\n    del got\n    log(f\"{len(screened)} candidates have both files on {probe[0]} and {probe[1]}\")\n\n    # 2. development quote volume; a candidate with a missing development kline file is dropped\n    vol, dropped_kline = {}, []\n    for i, s in enumerate(screened):\n        kl, _ = _fetch(arch, s, dev, metrics=False)\n        if any(v is None for v in kl.values()):\n            dropped_kline.append(s)\n            continue\n        vol[s] = float(sum(np.nansum(pdz.parse_klines(v)[\"quote_volume\"].to_numpy()) for v in kl.values()))\n        if (i + 1) % 25 == 0:\n            log(f\"  development volume: {i + 1}/{len(screened)} candidates\")\n    ranking = sorted(vol, key=lambda s: (-vol[s], s))\n    log(f\"{len(ranking)} candidates have every development kline file; {len(dropped_kline)} dropped\")\n\n    # 3. walk down the ranking; a symbol enters only if every warm-up and development file exists\n    universe, skipped, data = [], {}, {}\n    for s in ranking:\n        kl, mt = _fetch(arch, s, all_days)\n        mk = sum(v is None for v in kl.values()); mm = sum(v is None for v in mt.values())\n        if mk or mm:\n            skipped[s] = {\"missing_kline_files\": mk, \"missing_metrics_files\": mm}\n            continue\n        universe.append(s); data[s] = pdz.assemble(all_days, kl, mt)\n        if len(universe) == cfg[\"n_universe\"]:\n            break\n    if len(universe) < cfg[\"n_universe\"]:\n        raise SystemExit(f\"Only {len(universe)} symbols have complete files; the universe rule cannot be met. Stop.\")\n    log(f\"universe of {len(universe)} fixed; {len(skipped)} higher-ranked symbols skipped for missing files\")\n\n    # 4. thresholds from development bars only\n    n = len(all_days) * BPD\n    dev_mask = np.zeros(n, dtype=bool); dev_mask[len(warm) * BPD:] = True\n    day_id = np.clip(np.arange(n) // BPD - len(warm), 0, len(dev) - 1)\n    thresholds, coverage = {}, {}\n    S = np.zeros((len(universe), len(dev), 4)); N = np.zeros_like(S)\n    for i, s in enumerate(universe):\n        a = data[s]\n        pi, top = _states(a)\n        for name, sv in ((\"Pi\", pi), (\"top-trader state\", top)):\n            if np.isfinite(sv[dev_mask]).mean() < 0.5:\n                raise SystemExit(f\"{s}: {name} is defined on fewer than half of the development bars. Stop.\")\n        qv = a[\"quote_volume\"][dev_mask]; qv = qv[np.isfinite(qv)]\n        thresholds[s] = {\"primary\": ex.fit_thresholds(a[\"close\"], pi, dev_mask),\n                         \"rv\": ex.fit_thresholds(a[\"close\"], pi, dev_mask, rv_normalize=True),\n                         \"top\": ex.fit_thresholds(a[\"close\"], top, dev_mask),\n                         \"volume_terciles\": [float(x) for x in np.percentile(qv, [100 / 3, 200 / 3])]}\n        coverage[s] = {\"kline_bar_coverage\": a[\"kline_bar_coverage\"], \"metrics_bar_coverage\": a[\"metrics_bar_coverage\"],\n                       \"pi_defined_share_dev\": float(np.isfinite(pi[dev_mask]).mean())}\n        S[i], N[i] = ex.symbol_cells(a[\"close\"], pi, day_id, dev_mask, thresholds[s][\"primary\"], len(dev))\n    precision = ex.blinded_precision(S, N, universe, B=B)\n    return {\"spec\": \"P7S\", \"amendment\": \"A1\", \"stage\": \"development\", \"cfg\": cfg, \"universe\": universe,\n            \"development_quote_volume\": {s: vol[s] for s in ranking[: max(60, cfg[\"n_universe\"])]},\n            \"skipped_for_missing_files\": skipped, \"dropped_missing_development_kline\": dropped_kline,\n            \"n_candidates\": len(cands), \"n_screened\": len(screened), \"thresholds\": thresholds, \"coverage\": coverage,\n            \"blinded_precision_development\": precision}\n\n\ndef run_evaluation(cfg: dict, arch, dev_out: dict, log=print, B: int = ex.B_DEFAULT) -> dict:\n    universe, thresholds = dev_out[\"universe\"], dev_out[\"thresholds\"]\n    assert dev_out[\"cfg\"] == cfg, \"development outputs belong to another configuration\"\n    ev = pdz.days(cfg[\"eval_start\"], cfg[\"eval_end\"]); n_days = len(ev); step = cfg[\"shift_step_days\"]\n    if n_days % step != 0:\n        raise SystemExit(f\"{n_days} evaluation days is not a multiple of the shift step {step}. Stop.\")\n    hist = pdz.days(pdz.shift_day(cfg[\"eval_start\"], -HISTORY_DAYS), pdz.shift_day(cfg[\"eval_start\"], -1))\n    all_days = hist + ev\n    n = len(all_days) * BPD\n    eval_mask = np.zeros(n, dtype=bool); eval_mask[len(hist) * BPD:] = True\n    day_id = np.clip(np.arange(n) // BPD - len(hist), 0, n_days - 1)\n\n    # 1. download everything first; a failed download raises here, before anything is computed\n    for i, s in enumerate(universe):\n        _fetch(arch, s, all_days)\n        if (i + 1) % 5 == 0:\n            log(f\"  downloaded {i + 1}/{len(universe)} symbols\")\n\n    # 2. cells and bars for every variant\n    names = (\"primary\", \"rv\", \"top\", \"volume_low\", \"volume_mid\", \"volume_high\")\n    S = {v: np.zeros((len(universe), n_days, 4)) for v in names}\n    N = {v: np.zeros((len(universe), n_days, 4)) for v in names}\n    bars = {v: [] for v in names}\n    coverage = {}\n    for i, s in enumerate(universe):\n        kl, mt = _fetch(arch, s, all_days)\n        a = pdz.assemble(all_days, kl, mt)\n        del kl, mt\n        pi, top = _states(a)\n        thr = thresholds[s]\n        coverage[s] = {\"missing_kline_files\": a[\"missing_kline_files\"], \"missing_metrics_files\": a[\"missing_metrics_files\"],\n                       \"kline_bar_coverage_eval\": float(np.isfinite(a[\"close\"][eval_mask]).mean()),\n                       \"pi_defined_share_eval\": float(np.isfinite(pi[eval_mask]).mean())}\n        for v, sv, key in ((\"primary\", pi, \"primary\"), (\"rv\", pi, \"rv\"), (\"top\", top, \"top\")):\n            S[v][i], N[v][i] = ex.symbol_cells(a[\"close\"], sv, day_id, eval_mask, thr[key], n_days)\n            bars[v].append(ex.symbol_bars(a[\"close\"], sv, eval_mask, thr[key]))\n        terc = _volume_tercile(a[\"quote_volume\"], thr[\"volume_terciles\"])\n        base = bars[\"primary\"][-1]\n        for k, v in enumerate((\"volume_low\", \"volume_mid\", \"volume_high\")):\n            S[v][i], N[v][i] = ex.symbol_cells(a[\"close\"], pi, day_id, eval_mask & (terc == k), thr[\"primary\"], n_days)\n            bars[v].append({\"resp\": base[\"resp\"], \"state\": base[\"state\"],\n                            \"move\": np.where(terc[eval_mask] == k, base[\"move\"], 0).astype(np.int8)})\n\n    # 3. gate on event counts only\n    tot = N[\"primary\"].sum(1)\n    n_ok = int((tot >= ex.MIN_EVENTS).all(1).sum())\n    out = {\"spec\": \"P7S\", \"amendment\": \"A1\", \"stage\": \"evaluation\", \"cfg\": cfg, \"n_days\": n_days,\n           \"coverage\": coverage, \"symbols_with_enough_events\": n_ok,\n           \"events_per_symbol\": {s: {c: int(x) for c, x in zip(ex.CELLS, tot[i])} for i, s in enumerate(universe)}}\n    if n_ok < MIN_SYMBOLS_GATE:\n        out.update({\"status\": \"not_evaluable\", \"reason\": f\"only {n_ok} symbols have {ex.MIN_EVENTS} events in every \"\n                    f\"cell; the gate needs {MIN_SYMBOLS_GATE}. No statistic was computed.\"})\n        return out\n    log(f\"gate passed: {n_ok} symbols have at least {ex.MIN_EVENTS} events in every cell\")\n\n    # 4. statistics\n    primary = ex.evaluate(S[\"primary\"], N[\"primary\"], universe, bars=bars[\"primary\"], B=B, shift_step_days=step)\n    sens = {}\n    for v in names[1:]:\n        if int((N[v].sum(1) >= ex.MIN_EVENTS).all(1).sum()) < 2:\n            sens[v] = {\"status\": \"too few symbols with enough events\"}\n            continue\n        r = ex.evaluate(S[v], N[v], universe, bars=bars[v], B=SENS_B, shift_step_days=step)\n        sens[v] = {k: r[k] for k in (\"n_symbols_used\", \"events_per_cell_total\", \"beta_calm_mean\", \"beta_calm_ci90\",\n                                     \"beta_stress_mean\", \"did\", \"p_h2_stress_reveals\", \"verdict\")}\n    st = primary[\"shift_test\"]\n    by = dict(zip(st[\"shift_days\"], st[\"did_by_shift\"]))\n    out.update({\"status\": \"evaluated\", \"primary\": primary, \"sensitivity\": sens,\n                \"placebo_30_days\": {\"state_lagged_30_days\": by.get(30), \"state_led_30_days\": by.get(n_days - 30)},\n                \"readout\": readout(primary)})\n    return out\n\n\ndef readout(r: dict) -> str:\n    \"\"\"The fixed text of amendment A1, section 6, with numbers from the results.\"\"\"\n    x, (a, b), d, p, e = r[\"beta_calm_mean\"], r[\"beta_calm_ci90\"], r[\"did\"], r[\"p_h2_stress_reveals\"], r[\"E_margin\"]\n    if r[\"verdict\"] == \"supportive\":\n        return (\"In a pre-specified test on Binance perpetual futures, the response to ordinary price moves did not \"\n                \"depend on how close open interest was to its 30-day maximum (difference \"\n                f\"{x:.3f} standard deviations, 90% interval [{a:.3f}, {b:.3f}], inside the margin of \u00b1{e:.2f}), and \"\n                f\"the response to large moves did (difference-in-differences {d:.3f}, p = {p:.3f}). This is the \"\n                \"pattern Section 7 predicts. Open interest is a proxy for proximity to position limits. The test does \"\n                \"not identify the mechanism, and it does not test the identification theorem.\")\n    if r[\"verdict\"] == \"inconsistent\":\n        return (\"In a pre-specified test on Binance perpetual futures, the response to ordinary price moves depended \"\n                f\"on how close open interest was to its 30-day maximum (difference {x:.3f} standard deviations, at or \"\n                f\"beyond the margin of {e:.2f}). This is against the calm-silence prediction of Section 7.\")\n    return (\"A pre-specified test on Binance perpetual futures was inconclusive (calm difference \"\n            f\"{x:.3f}, 90% interval [{a:.3f}, {b:.3f}]; difference-in-differences {d:.3f}, p = {p:.3f}). It neither \"\n            \"supports nor contradicts the prediction of Section 7.\")\n"}
SHA = {"p7s_executor_A1.py": "7718ba33f55451d310bf9d9db01f08dbbd7754d5b5d9b887431e546a43824ab9", "p7s_data.py": "7e6e963acebe8aec6847877b8fa1b34e3a6aa83692b2075cbf23caf1c5993d36", "p7s_pipeline_A1.py": "5a85f51de705f286715d8fdc035d57dde963e2c19febc559cd7363a7c9856d7e"}
os.makedirs('p7s_code', exist_ok=True)
for name, text in SRC.items():
    got = hashlib.sha256(text.encode('utf-8')).hexdigest()
    assert got == SHA[name], f'{name}: source does not match the registered SHA-256'
    open(os.path.join('p7s_code', name), 'w', encoding='utf-8').write(text)
sys.path.insert(0, 'p7s_code')
import p7s_data as pdz, p7s_executor_A1 as ex, p7s_pipeline_A1 as pl
print('code verified:', {k: v[:12] for k, v in SHA.items()})
arch = pdz.Archive('p7s_cache')
CFG = pl.CFG_PRIMARY
t0 = time.time()

DEV_SHA = '7330ef3da46ea747e3b26dfd599ade3062f78d0f72741e901c0058dedbbeae43'
DEV_RECORD = {'universe': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', '1000PEPEUSDT', 'DOGEUSDT', 'XRPUSDT', 'BNBUSDT', 'ORDIUSDT', '1000SHIBUSDT', 'AVAXUSDT', 'ARBUSDT', 'FILUSDT', 'PEOPLEUSDT', 'BCHUSDT', 'LINKUSDT', 'MATICUSDT', 'ADAUSDT', 'TRBUSDT', '1000BONKUSDT', 'OPUSDT', 'ETCUSDT', 'LTCUSDT', '1000FLOKIUSDT', 'FTMUSDT', 'TIAUSDT', 'INJUSDT', 'DOTUSDT', 'SEIUSDT', 'APTUSDT', 'FETUSDT'], 'skipped': ['NEARUSDT', 'SUIUSDT', 'WLDUSDT'], 'events': {'calm_high': 140511, 'calm_low': 151319, 'stress_high': 6660, 'stress_low': 4481}, 'se_calm': '0.0068', 'se_did': '0.0945'}
if os.path.exists('p7s_eval_results.json'):
    raise SystemExit('The evaluation has already been run in this session. The results are in p7s_eval_results.json. It is not run twice.')

# Development outputs (second logged addition of 11 November 2026, before any evaluation data were read).
# If the file from the registered development run is in this session, it is used. If it is not, the registered
# development stage is run again here; it reads warm-up and development data only and is deterministic. Either
# way the outputs must reproduce what the registered development run printed, or the run stops.
dev_out, dev_source = None, None
if os.path.exists('p7s_dev_outputs.zip'):
    with zipfile.ZipFile('p7s_dev_outputs.zip') as z:
        raw_dev = z.read('p7s_dev_outputs.json')
    if hashlib.sha256(raw_dev).hexdigest() == DEV_SHA:
        dev_out, dev_source = json.loads(raw_dev), 'file with the registered fingerprint'
if dev_out is None:
    print('The development outputs of the registered run are not in this session.')
    print('Running the registered development stage again (warm-up and development data only) ...')
    dev_new = pl.run_development(CFG, arch, arch.list_symbol_folders())
    dev_new['utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    dev_new['code_sha256'] = SHA
    dev_new['downloads'] = {'downloaded': arch.n_downloaded, 'from_cache': arch.n_cached, 'absent_404': arch.n_absent}
    raw_dev = json.dumps(dev_new, indent=1, sort_keys=True, default=float).encode('utf-8')
    open('p7s_dev_outputs.json', 'wb').write(raw_dev)
    with zipfile.ZipFile('p7s_dev_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        z.write('p7s_dev_outputs.json')
    dev_out, dev_source = json.loads(raw_dev), 'regenerated in this session'
assert dev_out['code_sha256'] == SHA, 'The development stage ran with other code. Stop.'
_bp = dev_out['blinded_precision_development']
dev_seen = {'universe': list(dev_out['universe']), 'skipped': sorted(dev_out['skipped_for_missing_files']),
            'events': {k: int(v) for k, v in _bp['events_per_cell_total'].items()},
            'se_calm': '%.4f' % _bp['se_beta_calm'], 'se_did': '%.4f' % _bp['se_did']}
if dev_seen != DEV_RECORD:
    print('registered development run:', DEV_RECORD)
    print('this session              :', dev_seen)
    raise SystemExit('DEVELOPMENT OUTPUTS DO NOT MATCH the registered development run. Nothing was evaluated. Send this output; do not rerun.')
DEV_USED_SHA = hashlib.sha256(raw_dev).hexdigest()
print('development outputs:', dev_source, '| they reproduce the registered run | SHA-256', DEV_USED_SHA)

# Format gate (logged addition of 11 November 2026, made before any evaluation data were read). It checks the
# layout of a sample of evaluation files, one day a month for the two largest symbols, and can only stop the run.
# It computes no statistic and prints no price or open-interest value.
def _format_problem(sym, day):
    ku, mu = pdz.kline_url(sym, day), pdz.metrics_url(sym, day)
    got = arch.get_many([ku, mu])
    if got[ku] is None or got[mu] is None:
        return 'file missing'
    t0 = pdz.day_start_ms(day)
    try:
        k = pdz.parse_klines(got[ku]); m = pdz.parse_metrics(got[mu])
    except Exception as e:
        return 'parse error: ' + type(e).__name__
    ko = k['open_ms'].to_numpy(); mt = m['ts_ms'].to_numpy()
    if len(ko) < 280 or ko.min() < t0 or ko.max() >= t0 + 288 * pdz.BAR_MS or ((ko - t0) % pdz.BAR_MS).any():
        return 'kline time stamps are not on the 5-minute grid of the day'
    if not (k['close'].to_numpy() > 0).all():
        return 'kline close is not a positive number'
    if len(mt) < 250 or mt.min() < t0 or mt.max() >= t0 + 288 * pdz.BAR_MS:
        return 'metrics time stamps are outside the day'
    if (m['oi_value'].to_numpy() > 0).mean() < 0.9 or (m['top_ratio'].to_numpy() > 0).mean() < 0.9:
        return 'metrics values are missing'
    return None

gate_days = sorted({d for d in pdz.days(CFG['eval_start'], CFG['eval_end']) if d.endswith('-01')}
                   | {CFG['eval_start'], CFG['eval_end']})
gate = {(s, d): _format_problem(s, d) for s in dev_out['universe'][:2] for d in gate_days}
gate_fail = {f'{s} {d}': r for (s, d), r in gate.items() if r}
print(f'format gate: {len(gate) - len(gate_fail)} of {len(gate)} sampled files pass; problems: {gate_fail}')
if len(gate_fail) > 0.1 * len(gate):
    raise SystemExit('FORMAT GATE FAILED before any statistic was computed. Send this output; do not rerun.')

res = pl.run_evaluation(CFG, arch, dev_out)
res['format_gate'] = {'sampled': len(gate), 'problems': gate_fail}
res['utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
res['code_sha256'] = SHA
res['dev_outputs_sha256'] = DEV_USED_SHA
res['dev_outputs_source'] = dev_source
res['dev_outputs_registered_sha256'] = DEV_SHA
res['downloads'] = {'downloaded': arch.n_downloaded, 'from_cache': arch.n_cached, 'absent_404': arch.n_absent}
raw = json.dumps(res, indent=1, sort_keys=True, default=float).encode('utf-8')
open('p7s_eval_results.json', 'wb').write(raw)
with zipfile.ZipFile('p7s_eval_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('p7s_eval_results.json')

print('\n=== P7S A1 evaluation: the single registered run ===')
print('window:', CFG['eval_start'], 'to', CFG['eval_end'], '|', res['n_days'], 'days | symbols with enough events:',
      res['symbols_with_enough_events'])
if res['status'] != 'evaluated':
    print('NOT EVALUABLE:', res['reason'])
else:
    p = res['primary']
    print('symbols used:', p['n_symbols_used'], '| excluded:', p['symbols_excluded'])
    print('events per cell:', p['events_per_cell_total'])
    print('calm difference: %.4f | 90%% interval [%.4f, %.4f] | margin E = %.2f | H1 (calm silence): %s' % (
        p['beta_calm_mean'], p['beta_calm_ci90'][0], p['beta_calm_ci90'][1], p['E_margin'], p['h1_calm_silence']))
    print('stress difference: %.4f | DiD: %.4f | bootstrap 95%% interval [%.4f, %.4f]' % (
        p['beta_stress_mean'], p['did'], p['did_ci95'][0], p['did_ci95'][1]))
    print('H2 time-shift test: p = %.4f over %d shifts of %d days | H2 (stress reveals): %s' % (
        p['p_h2_stress_reveals'], p['shift_test']['n_shifts'], p['shift_test']['step_days'], p['h2_stress_reveals']))
    print('share of symbols with a positive stress difference: %.3f' % p['share_symbols_beta_stress_positive'])
    print('REGISTERED VERDICT:', p['verdict'])
    print('\nsensitivity checks (descriptive; none enters the verdict):')
    for k, v in res['sensitivity'].items():
        if 'did' in v:
            print('  %-12s calm %.4f | stress %.4f | DiD %.4f | shift p %.4f | symbols %d' % (
                k, v['beta_calm_mean'], v['beta_stress_mean'], v['did'], v['p_h2_stress_reveals'], v['n_symbols_used']))
        else:
            print('  %-12s %s' % (k, v['status']))
    print('  DiD with the state lagged 30 days:', res['placebo_30_days']['state_lagged_30_days'],
          '| led 30 days:', res['placebo_30_days']['state_led_30_days'])
    print('\nREADOUT (fixed text):')
    print(res['readout'])
print('SHA-256 of p7s_eval_results.json:', hashlib.sha256(raw).hexdigest())
print('downloads:', res['downloads'], '| minutes: %.1f' % ((time.time() - t0) / 60))
try:
    from google.colab import files
    files.download('p7s_eval_results.zip')
    files.download('p7s_dev_outputs.zip')
except Exception:
    pass
